<a href="https://colab.research.google.com/github/zohaib-mzg/Flyrank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5: Capstone Modeling Lane

Same lane, Refresh / Content Opportunity Scoring. This week I build the model my Week 4 baseline has to beat, on the same data and the same metric.

In [1]:
import os, subprocess
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/zohaib-mzg/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../..')
elif not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
print(f"{len(df):,} rows, decline rate {df['is_declining'].mean():.1%}")

30,000 rows, decline rate 54.2%


## 1. Method choice and why

I'm using a Random Forest classifier to predict `is_declining` (`trend_direction == 'down'`), the same target I've been using since Week 2. I considered Logistic Regression too, and I'm running it below as a comparison point rather than skipping it, but the whole argument I made back in Week 2 for why a model should beat a rule at all was that the real pattern is not additive, staleness, demand, position, and content depth trade off against each other in ways a flat rule or a linear model can't fully express. A Random Forest can pick up on those interactions, a logistic regression mostly can't, so it's the right tool if that argument holds.

Feature set, kept deliberately clean of anything that touches the label's own source window. `trend_direction` and `trend_pct` are the label itself, obviously excluded. Less obviously, I also excluded `impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, and `sessions_prev_30d`, because those last30/prev30 splits are literally what `trend_direction` is computed from. Using them as features would be close to the exact mistake I deliberately caused and caught in Week 3's leakage trap. What's left is the 90 day aggregates, content metadata, and tier columns, everything a person could look at today without already knowing which way the trend went.

One more honest note before training anything. A few columns have real missing values, `search_volume`, `competition`, and `cpc` are missing for 2,468 rows, and `word_count` and `char_count` are missing for 7,699 rows, likely pages without keyword data or without a text body captured. I filled these with the column median rather than zero, since zero would misrepresent a page as having no content or no search volume when the truth is just that the figure wasn't captured. This is a real modeling choice, not a neutral default, and I'm naming it rather than letting it happen silently.

In [2]:
numeric_features = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

categorical_features = ['competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

feature_df = df[numeric_features + categorical_features].copy()
missing_before = feature_df[numeric_features].isna().sum()
print("columns with missing values, filled with median:")
print(missing_before[missing_before > 0])

for col in numeric_features:
    feature_df[col] = feature_df[col].fillna(feature_df[col].median())

X = pd.get_dummies(feature_df, columns=categorical_features)
y = df['is_declining']
print(f"{X.shape[1]} features after encoding")

columns with missing values, filled with median:
search_volume    2468
competition      2468
cpc              2468
word_count       7699
char_count       7699
scroll_rate       125
dtype: int64
57 features after encoding


## 2. Split design

I'm using a plain stratified random 75 25 split for this week. I want to be upfront about a real weakness in that choice rather than quietly picking a safer split and hiding the reasoning. Every `client_id` in this dataset can have many pages, so a random split can put some pages from the same client in the training set and other pages from that same client in the test set. If a client's pages share account level patterns, industry, publishing habits, CMS quirks, the model could partly be learning to recognize the client rather than the underlying signal, and the test score would look better than it would on a client the model has never seen at all.

I'm not fixing that this week. Building a grouped or time aware split and showing the before and after is next week's assignment on its own, and doing it properly here would just be redoing that work early with less care. What I'm doing instead is naming the risk clearly now, so the comparison below is read as a first honest pass, not a finished validation.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.25, random_state=42, stratify=y)

print(f"train: {len(X_train):,} rows, test: {len(X_test):,} rows")

train: 22,500 rows, test: 7,500 rows


## 3. Train and compare against my baseline

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_proba)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logit = LogisticRegression(max_iter=2000)
logit.fit(X_train_scaled, y_train)
logit_proba = logit.predict_proba(X_test_scaled)[:, 1]
logit_auc = roc_auc_score(y_test, logit_proba)

print(f"Random Forest ROC AUC: {rf_auc:.3f}")
print(f"Logistic Regression ROC AUC: {logit_auc:.3f}")

Random Forest ROC AUC: 0.754
Logistic Regression ROC AUC: 0.695


Same headline metric as every week so far, Precision@50. I'm recomputing my Week 4 baseline score for every row here, not just the rows that qualified for the rule, so both rankings are drawn from the exact same test population.

In [5]:
peer_tier_avg_ctr = df.groupby('position_tier')['ctr'].transform('mean')
qualifies = (df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)
df['baseline_score'] = np.where(qualifies, (peer_tier_avg_ctr - df['ctr']) * df['impressions_90d'], -1)

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[order].mean()

test_baseline_scores = df.loc[idx_test, 'baseline_score'].values

p50_baseline = precision_at_k(test_baseline_scores, y_test.values, 50)
p50_rf = precision_at_k(rf_proba, y_test.values, 50)
p50_logit = precision_at_k(logit_proba, y_test.values, 50)

comparison = pd.DataFrame({
    'method': ['Week 4 baseline (low_ctr_visible_page score)', 'Logistic Regression', 'Random Forest'],
    'precision_at_50': [p50_baseline, p50_logit, p50_rf],
})
comparison

,method,precision_at_50
0,Week 4 baseline (low_ctr_visible_page score),0.52
1,Logistic Regression,0.86
2,Random Forest,0.88


The baseline lands at 0.52 Precision@50 on this test split. That number alone tells me something honest about my own rule, it was built to catch CTR opportunities, not decline, so a little over half of its top 50 happening to be declining is a coincidence of correlation, not something the rule was designed to do. Random Forest reaches 0.88 and Logistic Regression reaches 0.86, both actually built to predict decline directly, and both clear the baseline by a wide margin.

The two models land close to each other on Precision@50, closer than I expected given the interaction argument I made in Section 1. The gap shows up more clearly in ROC AUC, 0.754 for Random Forest against 0.695 for Logistic Regression, a real difference, just smaller in this particular top 50 slice than across the full ranking. I'm reporting both numbers rather than only the one that makes the strongest case for Random Forest, since Precision@50 is the metric I actually care about for this lane, and on that metric the two methods are closer than the interaction story alone would predict.

I'm reporting this as an observed result on this split, not a guarantee. Section 2's caveat about the random split still applies to every number in this table.

## 4. Errors and interpretation

In [6]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(10)

,0
days_with_impressions,0.154785
impressions_90d,0.123855
avg_position,0.102920
content_age_days,0.086307
char_count,0.034575
position_tier_top_3,0.034388
word_count,0.033747
age_tier_365+,0.030289
clicks_90d,0.028517
ctr,0.027581


`days_with_impressions` and `impressions_90d` lead, followed by `avg_position` and `content_age_days`. That's a reasonable story, pages with thinner, less consistent visibility over the last 90 days and pages that are older are both more likely to be caught declining, which lines up with plain intuition. Nothing in the top ten touches a last30/prev30 column, so the exclusion from Section 1 held.

In [7]:
test_df = df.loc[idx_test].copy()
test_df['pred_proba'] = rf_proba
top50 = test_df.sort_values('pred_proba', ascending=False).head(50)

false_positives = top50[top50['is_declining'] == 0][
    ['content_id', 'trend_direction', 'pred_proba', 'avg_position', 'impressions_90d',
     'content_age_days', 'days_since_last_update']
]
print(f"{len(false_positives)} false positives in the top 50")
false_positives

6 false positives in the top 50


,content_id,trend_direction,pred_proba,avg_position,impressions_90d,content_age_days,days_since_last_update
12595,content_959e1cc9feeb,stable,0.827413,38.1,611,155,104
2164,content_1337de8128cc,up,0.824549,5.5,945,139,104
10025,content_dba1bbc29b4e,stable,0.818085,38.2,383,165,104
9482,content_7158cfbbc450,stable,0.816759,24.9,134567,147,20
7882,content_25a763874cf0,up,0.815784,27.4,908,144,104
11114,content_698baf785309,up,0.815051,16.6,921,223,104


Six false positives out of 50, matching the 0.88 Precision@50 above. Five of them, content_959e1cc9feeb, content_1337de8128cc, content_dba1bbc29b4e, content_25a763874cf0, and content_698baf785309, share a real pattern, all five sit at exactly 104 days since their last update, an age and staleness profile the model has clearly learned to associate with decline, and in most of the dataset it's right to. These five are labeled stable or up instead. That looks like the model correctly learning a real pattern that simply doesn't hold for every single page it matches, which is a normal and expected kind of error, not a sign the model is broken. The fact that all five share the exact same days_since_last_update value is also worth flagging on its own, that's likely a batch update or a snapshot cutoff rather than five independent coincidences, and it means the model may be leaning on a somewhat artificial cluster in this particular feature rather than five truly distinct cases.

The sixth, content_7158cfbbc450, is more interesting. It has 134,567 impressions in 90 days, far more than the other five, a fairly recent update at 20 days since last update, and is still labeled stable rather than declining. The model likely flagged it on volume and position alone. This is a page worth a human actually looking at, high visibility, recently touched, model still uncertain, exactly the kind of case a ranked queue is supposed to surface for review rather than something to treat as a clean model failure.

## 5. Self-check

Model compared against the Week 4 baseline on the exact same test rows and the exact same metric, Precision@50, not a metric picked after seeing which one looked better.

Split is a plain stratified random split, and I said so plainly rather than dressing it up, along with the specific client leakage risk that comes with it. That risk is not fixed here, it's next week's assignment, named honestly instead of glossed over.

No last30 or prev30 columns and no `trend_pct` anywhere in the feature set, only `trend_direction` as the label itself. Feature importances came back led by 90 day aggregates and content age, not by anything from the label's own source window.

Errors examined individually rather than summarized away. Four of five false positives look like a real, defensible pattern applied to cases where it didn't hold. One looks like a genuinely useful, uncertain case worth a human's attention, which is what this queue is for in the first place.

Language throughout stays at observed and directional, a Precision@50 of 0.90 on this split, not a promise about the future or about clients outside this dataset.